In [1]:
import pandas as pd
import numpy as np
import re
import csv
import os
import torch
import esm
from Bio import SeqIO
from evcouplings.compare import DistanceMap
from forecast_utils import *

### 2021

In [2]:
from forecast_utils import PrepTools, MutationTools, FeaturePipeline, CatalogNormalizer

# 1) Load protein sequences
protein_sequences_df = PrepTools.load_protein_sequences("./data/catalog/protein_sequences.csv")
genes_of_interest = protein_sequences_df['gene'].unique()

# 2) Load + standardize WHO catalog
standardized_catalog = CatalogNormalizer.load_and_standardize(
    "./data/catalog/WHO-UCN-GTB-PCI-2021.7-eng.xlsx",
    year=2021,
    genes_of_interest=genes_of_interest
)

# # 3) Save preprocessed
# PrepTools.save_preprocessed(
#     standardized_catalog,
#     "./data/derived_features/2021/2021_mutations_with_one_letter_all_confidence.csv"
# )

# # 4) Generate mutated FASTAs
# results, mismatches = MutationTools.generate_mutated_fastas(
#     protein_sequences_df,
#     standardized_catalog,
#     output_dir="./mutated_sequences_2021"
# )
# print(f"{len(results)} mutations applied OK")
# print(f"{len(mismatches)} problems logged")

In [3]:


# 5) Run full feature pipeline
final_df = FeaturePipeline.run(
    catalog_df=standardized_catalog,
    protein_df=protein_sequences_df,
    fasta_dir="./mutated_sequences_2021",
    distmap_dir="./data/distmaps/",
    protein_details_path="./data/catalog/17_proteins_details.xlsx",                    
    rosetta_dir="./data/Rosetta/refined",
    aaindex_path="./data/catalog/AAIndex_PCA.csv",
    esm_model_path="/datasets/bio/esm/models/esm2_t6_8M_UR50D.pt",
    out_dir="./data/derived_features/2021",
    top_k_llr=10
)

print(final_df.shape)
print(final_df.head())



 Feature pipeline finished. Final dataset saved at ./data/derived_features/2021/step6_aaindex.csv
(4709, 50)
   fa_atr  fa_rep  fa_sol  fa_elec  fa_dun  thermostability Wildtype_AA  \
0  -2.890   0.862   0.112    0.275   0.308         1.921217           S   
1  -2.890   0.862   0.112    0.275   0.308         1.921217           S   
2  -0.436   0.001   0.517   -0.290   0.815         1.104466           S   
3  -0.436   0.001   0.517   -0.290   0.815         1.104466           S   
4   0.417  -0.004  -0.581    0.346  -1.518         2.265218           D   

   position Mutated_AA mutation  ...  mut_AAIndex4 delta_AAIndex4  \
0         2          I      S2I  ...      4.249276       7.334761   
1         2          I      S2I  ...      4.249276       7.334761   
2         2          R      S2R  ...      0.054632       3.140117   
3         2          R      S2R  ...      0.054632       3.140117   
4         5          G      D5G  ...     -7.335674      -5.650224   

  mut_AAIndex5 delta_AAIn

In [4]:
rename_map = {
    # Mutation info
    "one_letter_mutation": "mutation_oneletter",
    "Wildtype_AA": "mutation_wt",
    "position": "mutation_pos",
    "Mutated_AA": "mutation_mut",
    # Frequency
    "frequency": "freq_variant",
    # Rosetta
    "fa_atr": "Rosetta_fa_atr",
    "fa_rep": "Rosetta_fa_rep",
    "fa_sol": "Rosetta_fa_sol",
    "fa_elec": "Rosetta_fa_elec",
    "fa_dun": "Rosetta_fa_dun",
    "thermostability": "Rosetta_ddG",
    # Delta-Z
    "delta_z": "DeltaZ",
    # Proximity
    "WHO_Adjusted_Position": "Prox_WHO_Adjusted_Pos",
    "Proximity_1D": "Prox_1D",
    "Nearest_1D_Index": "Prox_1D_nearest",
    "Proximity_to_R_Conferring": "Prox_3D",
    "Nearest_Mutation_Index": "Prox_3D_nearest",
    "Proximity_to_R_Conferring_zeroed": "Prox_3D_zeroed",
    # LLR
    "llr_score": "LLR_score",
}

# Apply rename
final_df = final_df.rename(columns=rename_map)

# Expand renaming for AAIndex
for i in range(1, 9):
    final_df = final_df.rename(columns={
        f"mut_AAIndex{i}": f"AAIndex_mut{i}",
        f"delta_AAIndex{i}": f"AAIndex_delta{i}"
    })

# Expand renaming for LLR expanded dims
for c in final_df.columns:
    if c.startswith("expanded_llr_dim_"):
        dim = c.split("_")[-1]
        final_df = final_df.rename(columns={c: f"LLR_dim{dim}"})

# Define ordered groups
ordered_cols = [
    "gene", "drug", "confidence", "phenotype",
    "mutation_oneletter", "mutation_wt", "mutation_pos", "mutation_mut",
    "freq_variant",
    "Rosetta_fa_atr", "Rosetta_fa_rep", "Rosetta_fa_sol", "Rosetta_fa_elec",
    "Rosetta_fa_dun", "Rosetta_ddG",
    "DeltaZ",
    "Prox_WHO_Adjusted_Pos", "Prox_1D", "Prox_1D_nearest",
    "Prox_3D", "Prox_3D_nearest", "Prox_3D_zeroed",
    "LLR_score"
] + sorted([c for c in final_df.columns if c.startswith("LLR_dim")],
           key=lambda x: int(x.replace("LLR_dim",""))) \
  + sorted([c for c in final_df.columns if c.startswith("AAIndex_")])

# Reorder (and keep extras at end if any)
final_df = final_df[[c for c in ordered_cols if c in final_df.columns] +
                    [c for c in final_df.columns if c not in ordered_cols]]


In [6]:
final_df.to_csv('./data/derived_features/2021/2021_final_df.csv',index=False)

### 2023

In [7]:
from forecast_utils import CatalogNormalizer, MutationTools, PrepTools

# 1) Load proteins
protein_sequences_df = PrepTools.load_protein_sequences("./data/catalog/protein_sequences.csv")
genes_of_interest = protein_sequences_df['gene'].unique()

# 2) Standardize catalog (2023)
cat2023 = CatalogNormalizer.load_and_standardize(
    "./data/catalog/WHO-UCN-TB-2023.7-eng.xlsx",
    year=2023,
    genes_of_interest=genes_of_interest
)

# 3) Save standardized output
cat2023.to_csv("./data/derived_features/2023/2023_mutations_with_one_letter_all_confidence.csv", index=False)

# 4) Generate mutated FASTAs
results, mismatches = MutationTools.generate_mutated_fastas(
    protein_sequences_df,
    cat2023,
    output_dir="mutated_sequences_2023"
)

print(f"{len(results)} mutations applied OK")
print(f"{len(mismatches)} problems logged")


/work/pi_annagreen_umass_edu/mahbuba/esmfold/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


6172 mutations applied OK
0 problems logged


In [8]:


# 5) Run full feature pipeline
final_df = FeaturePipeline.run(
    catalog_df=cat2023,
    protein_df=protein_sequences_df,
    fasta_dir="./mutated_sequences_2023",
    distmap_dir="./data/distmaps/",
    protein_details_path="./data/catalog/17_proteins_details.xlsx",                    
    rosetta_dir="./data/Rosetta/refined",
    aaindex_path="./data/catalog/AAIndex_PCA.csv",
    esm_model_path="/datasets/bio/esm/models/esm2_t6_8M_UR50D.pt",
    out_dir="./data/derived_features/2023",
    top_k_llr=10
)

print(final_df.shape)
print(final_df.head())



 Feature pipeline finished. Final dataset saved at ./data/derived_features/2023/step6_aaindex.csv
(6163, 52)
   fa_atr  fa_rep  fa_sol  fa_elec  fa_dun  thermostability Wildtype_AA  \
0  -2.890   0.862   0.112    0.275   0.308         1.921217           S   
1  -2.890   0.862   0.112    0.275   0.308         1.921217           S   
2  -0.436   0.001   0.517   -0.290   0.815         1.104466           S   
3  -0.436   0.001   0.517   -0.290   0.815         1.104466           S   
4   0.417  -0.004  -0.581    0.346  -1.518         2.265218           D   

   position Mutated_AA mutation_x  ...  mut_AAIndex4 delta_AAIndex4  \
0         2          I        S2I  ...      4.249276       7.334761   
1         2          I        S2I  ...      4.249276       7.334761   
2         2          R        S2R  ...      0.054632       3.140117   
3         2          R        S2R  ...      0.054632       3.140117   
4         5          G        D5G  ...     -7.335674      -5.650224   

  mut_AAIndex

In [9]:
rename_map = {
    # Mutation info
    "one_letter_mutation": "mutation_oneletter",
    "Wildtype_AA": "mutation_wt",
    "position": "mutation_pos",
    "Mutated_AA": "mutation_mut",
    # Frequency
    "frequency": "freq_variant",
    # Rosetta
    "fa_atr": "Rosetta_fa_atr",
    "fa_rep": "Rosetta_fa_rep",
    "fa_sol": "Rosetta_fa_sol",
    "fa_elec": "Rosetta_fa_elec",
    "fa_dun": "Rosetta_fa_dun",
    "thermostability": "Rosetta_ddG",
    # Delta-Z
    "delta_z": "DeltaZ",
    # Proximity
    "WHO_Adjusted_Position": "Prox_WHO_Adjusted_Pos",
    "Proximity_1D": "Prox_1D",
    "Nearest_1D_Index": "Prox_1D_nearest",
    "Proximity_to_R_Conferring": "Prox_3D",
    "Nearest_Mutation_Index": "Prox_3D_nearest",
    "Proximity_to_R_Conferring_zeroed": "Prox_3D_zeroed",
    # LLR
    "llr_score": "LLR_score",
}

# Apply rename
final_df = final_df.rename(columns=rename_map)

# Expand renaming for AAIndex
for i in range(1, 9):
    final_df = final_df.rename(columns={
        f"mut_AAIndex{i}": f"AAIndex_mut{i}",
        f"delta_AAIndex{i}": f"AAIndex_delta{i}"
    })

# Expand renaming for LLR expanded dims
for c in final_df.columns:
    if c.startswith("expanded_llr_dim_"):
        dim = c.split("_")[-1]
        final_df = final_df.rename(columns={c: f"LLR_dim{dim}"})

# Define ordered groups
ordered_cols = [
    "gene", "drug", "confidence", "phenotype",
    "mutation_oneletter", "mutation_wt", "mutation_pos", "mutation_mut",
    "freq_variant",
    "Rosetta_fa_atr", "Rosetta_fa_rep", "Rosetta_fa_sol", "Rosetta_fa_elec",
    "Rosetta_fa_dun", "Rosetta_ddG",
    "DeltaZ",
    "Prox_WHO_Adjusted_Pos", "Prox_1D", "Prox_1D_nearest",
    "Prox_3D", "Prox_3D_nearest", "Prox_3D_zeroed",
    "LLR_score"
] + sorted([c for c in final_df.columns if c.startswith("LLR_dim")],
           key=lambda x: int(x.replace("LLR_dim",""))) \
  + sorted([c for c in final_df.columns if c.startswith("AAIndex_")])

# Reorder (and keep extras at end if any)
final_df = final_df[[c for c in ordered_cols if c in final_df.columns] +
                    [c for c in final_df.columns if c not in ordered_cols]]


In [10]:
final_df

,gene,drug,confidence,phenotype,mutation_oneletter,mutation_wt,mutation_pos,mutation_mut,freq_variant,Rosetta_fa_atr,...,AAIndex_mut2,AAIndex_mut3,AAIndex_mut4,AAIndex_mut5,AAIndex_mut6,AAIndex_mut7,AAIndex_mut8,mutation_x,mutation_y,effect
0,Rv0678,Bedaquiline,3) Uncertain significance,Unknown,S2I,S,2,I,0.75,-2.890,...,5.857561,2.403990,4.249276,-2.484487,-2.770203,3.875806,-0.357405,S2I,p.Ser2Ile,missense_variant
1,Rv0678,Clofazimine,3) Uncertain significance,Unknown,S2I,S,2,I,0.00,-2.890,...,5.857561,2.403990,4.249276,-2.484487,-2.770203,3.875806,-0.357405,S2I,p.Ser2Ile,missense_variant
2,Rv0678,Bedaquiline,3) Uncertain significance,Unknown,S2R,S,2,R,0.00,-0.436,...,-15.801100,-1.267952,0.054632,-8.122787,-8.346795,-4.802642,-5.875155,S2R,p.Ser2Arg,missense_variant
3,Rv0678,Clofazimine,3) Uncertain significance,Unknown,S2R,S,2,R,0.00,-0.436,...,-15.801100,-1.267952,0.054632,-8.122787,-8.346795,-4.802642,-5.875155,S2R,p.Ser2Arg,missense_variant
4,Rv0678,Bedaquiline,3) Uncertain significance,Unknown,D5G,D,5,G,0.00,0.417,...,21.629177,3.123584,-7.335674,-9.426146,8.832837,-4.847755,-3.544305,D5G,p.Asp5Gly,missense_variant
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6158,tlyA,Capreomycin,3) Uncertain significance,Unknown,L241P,L,241,P,1.00,1.631,...,11.882400,-16.928919,19.220780,6.085233,-1.032870,-4.807777,0.106104,L241P,p.Leu241Pro,missense_variant
6159,tlyA,Capreomycin,3) Uncertain significance,Unknown,R249Q,R,249,Q,0.00,1.709,...,-9.140128,1.733513,-0.064256,1.715814,-2.134583,-1.499761,0.578000,R249Q,p.Arg249Gln,missense_variant
6160,tlyA,Capreomycin,3) Uncertain significance,Unknown,S252L,S,252,L,0.00,-3.248,...,3.086893,11.307291,6.859719,-2.107819,2.222849,-0.398988,0.928746,S252L,p.Ser252Leu,missense_variant
6161,tlyA,Capreomycin,3) Uncertain significance,Unknown,A259T,A,259,T,0.00,-2.932,...,5.971147,-0.063089,-0.562307,-1.275904,-7.649776,4.139779,2.479809,A259T,p.Ala259Thr,missense_variant


In [11]:
final_df.to_csv('./data/derived_features/2023/2023_final_df.csv',index=False)